In [ ]:
import json
import re
from pathlib import Path


## Task 1 File ingestion and regular expressions

Function `read_file()`uses the Path to read a file, it is written to handel utf-8 or ISO-8859-1/Latin-1 encoding. It returns a tuple containing the files content and the used encoding as strings.

In [ ]:
def read_file(input_file: Path) -> tuple[str, str]:
    """Read a file and return its content and the encoding used.
    
    Arguments:
        input_file: Path to the file to read.
        
    Returns:
        A tuple containing the file content as a string and the encoding used as a string."""
    
    try:
        # Try to read the file with UTF-8 encoding
        with input_file.open(encoding="utf-8") as file:
            file_text_content = file.read()
            encoding_used = "utf-8"

    except UnicodeDecodeError:
        # If UTF-8 fails, try ISO-8859-1/Latin-1 encoding
        with input_file.open(encoding="ISO-8859-1") as file:
            file_text_content = file.read()
            encoding_used = "ISO-8859-1"

    # Returns a tuple with file_text_content and encoding_used
    return file_text_content, encoding_used


The function `parse_log()` parses the log file contents and returns the valid transactions as a list of dictionaries. 
The functions uses regular expresions to filter out the valid transactions. The regex pattern groups the different elements together and names each group. The different groups are then organised as dictionary keys. The filterd out groups are: timestamp, register_id, item, price, currency and status.

In [ ]:
def parse_log(file_text_content: str) -> list[dict[str, str | float]]:
    """Parse log file content and and return valid transactions.

    A parser function analyses text and extracts structured information from it according to a defined format.

    Arguments:
        file_text_content: The complete text content of a log file.

    Returns:
        A list of dictionaries containing valid transactions.
        Each dictionary contains the timestamt, register ID, item,
        price, currency, and status. The price is stored as a float.
        Lines that do not match the expected log format are ignored.
    """

    # Regex pattern for valid transactions
    # () - groups result together
    # ?P<...> names the group
    # r"..." raw string
    pattern = (
    r"\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\] " 
    r"\[(?P<register_id>\w+)\] " 
    r"item=(?P<item>\S+) " 
    r"price=(?P<price>\d+\.\d+) "  
    r"(?P<currency>EUR|NOK|USD) " 
    r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)" 
) 
    # empty list to collect the valid lines in the logfiles
    valid_transactions = []

    # splits up the file so we can work with it line by line
    for line in file_text_content.splitlines():

        # check whether this line matches the pattern
        match = re.fullmatch(pattern, line)

        # if the line is valid, extract this field
        if match:
            transaction = match.groupdict()

            # convert prise to float
            transaction["price"] = float(transaction["price"])

            # add the transaction to the list
            valid_transactions.append(transaction)

    return valid_transactions

## Task 2 recursive utility functions

The code `find_log_files()` is used to find .log files in a directory. It takes in a path to the startingpoint of the directory. Then it recursively searches down the path until it finds the .log files. It returns a list of path objects representing the log files it finds. 

In [ ]:
def find_log_files(directory: Path) -> list[Path]:
    # directory: Path - funksjonen mottar en mappebane
    # -> list[Path] - funksjonen skal returnere en liste med filbaner - ikke filinnholdet
    """Find all log files in a directory and its subdirectories.
    
    The function searches recursively through the directory structure 
    and collects the paths of all files ending in .log.
    
    Arguments:
        directory: Path to the directory where the search begins.
        
    Returns:
        A list of Path objects representing the log files found.
    """

    all_log_files = []

    for item in directory.iterdir():

        if item.is_dir():
            # Search the subdirectory recursively and collect its log files

            all_log_files.extend(find_log_files(item))  # This calls the function again for subfolders 

        elif item.is_file():
            # check file ending - .suffix gives fileending including . 
            if item.suffix == ".log":
                all_log_files.append(item)

    return all_log_files

The function `find_active_registers()` searches nested dictionaries and lists to find active register ids. This function is used on a the store_config.json file, witch gives information about the configuration of some of the stores in the company. 

In [ ]:
def find_active_registers(data: dict | list) -> set[str]:
    """Find IDs fo all active registers in nested data.
    
    Arguments:
        data: nested dictionaries and lists to search.
        
    Returns:
        A set of IDs belonging to active registers"""

    # empty set for collecting active registers
    active_registers = set()

    if isinstance(data, dict):
    # isinstance(item, dict) asks if the item a dictionary
        # Check wheter this dictionary describes an active register
        if (
            "id" in data
            and data.get("active") is True
        ):
            active_registers.add(data["id"])

        # Search recursively through the dictionary values
        for value in data.values():
            # data.values() gives the values inside the dict
            # each value gets sendt back to the same function
            # if it is a dict og a list the function "searches" it
            # .update() adds all the found ids into the existing set
            active_registers.update(find_active_registers(value))

    elif isinstance(data, list):
    # isinstance(item, list) asks if the item a list
        # Search recursively through each element in the list
        for item in data:
            active_registers.update(find_active_registers(item))

    return active_registers

## Task 5 Create full workflow

### Recursively find all log files and extract active register IDs

Here we find all of the log files starting with the path "data/2026". We use the recursive function to go through the directory.

In [ ]:
data_dictionary = Path("data/2026")
log_files = find_log_files(data_dictionary)

print(f"Number of log files found: {len(log_files)}")

Here we find the active register ids from the store config file. Since the file did not have information about all of the stores we chose to just do the analysis about the active register and extracting the information, but we did not use it to filter out any of the data from the transaction logs.

In [ ]:
# Store confuguration - data from store-config.json
config_path = Path("data/store_config.json")

config_text, encoding_used = read_file(config_path)
# json.loads konverts text into python datastructure with dictionaries and lists
# loads - string of text, while load takes a fileobject
config_data = json.loads(config_text)

# Find all active register IDs
active_registers = find_active_registers(config_data)

# Print register information by store in config file
for region in config_data["regions"]:
    for store in region["stores"]:
        print(store["name"])

        for register in store["registers"]:
            if register["active"]:
                status = "Active"
            else:
                status = "Inactive"

            print(f"{register["id"]}, {status}")

# sorted sorts the set alfabeticly
print(f"Active register IDs form config: {sorted(active_registers)}")

### Ingest text logs and parse records into a list of dictionaries

In this step, we process each log file individually and collect all valid transactions into a single list. First, we create an empty list called `all_transactions`. We then loop through the log files in sorted order to ensure a consistent processing order.

For each file, `read_file()` reads the text content and handles the file encoding. The content is then passed to `parse_log()`, which extracts valid transaction records and returns them as a list of dictionaries. Each dictionary represents one transaction and contains the relevant information, such as the timestamp, register ID, item, price, currency, and status.

The `extend()` method adds all transactions from the current file to `all_transactions`. This allows us to combine the records from all nine log files into one list for further processing. Finally, we print the total number of transactions collected. The result is 720 transactions.

In [ ]:
# Create empty list for all transactions
all_transactions = []

# Read and parse each log file
# Collect all transactions into one combined list
for log_file in sorted(log_files):
    file_text_content, encoding_used = read_file(log_file)
    transactions = parse_log(file_text_content)
    all_transactions.extend(transactions)

print(f"Number of transactions: {len(all_transactions)}")

### Build the initial Pandas DataFrame

In this step, we convert the list of transaction dictionaries into a Pandas DataFrame. We then inspect the first 20 rows, check the data types and missing values, verify the total number of transactions, and count the transactions from each register.

In [ ]:
# Task 5.3 Build the initial DataFrame
df = pd.DataFrame(all_transactions)

# Check the DataFrame
print(df.head(20))
df.info()    # Gives info about the DataFrame
print(f"Number of transactions: {len(df)}")
print(df["register_id"].value_counts())